# Module 15 — Logistic Regression Practice Problems
### Titanic Survival Prediction

এই notebook-এ Module 15-এর প্রতিটি topic-এর উপর হাতে-কলমে practice করবে।
প্রতিটি problem-এর নিচে code cell-এ solution লিখো।

**Dataset:** `titanic_data_updated.csv`

---
### Problem 1 — Imports and Data Loading

নিচের সব library import করো:
- `numpy`, `pandas`
- `sklearn` থেকে: `train_test_split`, `SimpleImputer`, `OrdinalEncoder`, `OneHotEncoder`, `LabelEncoder`, `StandardScaler`, `MinMaxScaler`, `Pipeline`, `ColumnTransformer`, `LogisticRegression`

`titanic_data_updated.csv` লোড করো `df` নামে এবং প্রথম 5টি row দেখাও।

In [15]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

In [16]:
df = pd.read_csv('/content/Titanic-Dataset.csv')
df.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


---
### Problem 2 — Feature Engineering

দুটি নতুন column তৈরি করো:

1. `Family_Size` — formula: `SibSp + Parch + 1`
2. `Deck` — `Cabin` column-এর NaN গুলো `"Missing"` দিয়ে fill করো, তারপর প্রথম character নিয়ে `Deck` column তৈরি করো

শেষে `df.sample(5)` দিয়ে result দেখাও।

In [17]:
# YOUR CODE HERE

df['Family_Size'] = df['SibSp'] + df['Parch'] + 1
df['Deck'] = df['Cabin'].fillna('Missing').astype(str).str[0]
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
828,829,1,3,"McCormack, Mr. Thomas Joseph",male,NaN,0,0,367228,7.7500,NaN,Q,1,M
816,817,0,3,"Heininen, Miss. Wendla Maria",female,23.0,0,0,STON/O2. 3101290,7.9250,NaN,S,1,M
843,844,0,3,"Lemberopolous, Mr. Peter L",male,34.5,0,0,2683,6.4375,NaN,C,1,M
337,338,1,1,"Burns, Miss. Elizabeth Margaret",female,41.0,0,0,16966,134.5000,E40,C,1,E
240,241,0,3,"Zabour, Miss. Thamine",female,NaN,1,0,2665,14.4542,NaN,C,2,M


---
### Problem 3 — X and y Split

`df` থেকে features এবং target আলাদা করো:

- `X` = `Survived` বাদে সব column
- `y` = শুধু `Survived` column

`X.shape` এবং `y.shape` print করো।

In [26]:
# YOUR CODE HERE
X = df.drop(columns = ['Survived'])
y = df['Survived']

X.shape, y.shape

((891, 13), (891,))

In [30]:
X.sample(5)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
679,680,1,"Cardeza, Mr. Thomas Drake Martinez",male,36.0,0,1,PC 17755,512.3292,B51 B53 B55,C,2,B
391,392,3,"Jansson, Mr. Carl Olof",male,21.0,0,0,350034,7.7958,NaN,S,1,M
554,555,3,"Ohman, Miss. Velin",female,22.0,0,0,347085,7.7750,NaN,S,1,M
122,123,2,"Nasser, Mr. Nicholas",male,32.5,1,0,237736,30.0708,NaN,C,2,M
432,433,2,"Louch, Mrs. Charles Alexander (Alice Adelaide ...",female,42.0,1,0,SC/AH 3085,26.0000,NaN,S,2,M


In [31]:
y.sample(5)

,Survived
564,0
614,0
643,1
494,0
46,0


---
### Problem 4 — Train-Test Split

`train_test_split` ব্যবহার করে `X_train`, `X_test`, `y_train`, `y_test` তৈরি করো।

Parameters:
- `test_size = 0.2`
- `random_state = 42`
- `stratify = y`

`X_train` এবং `X_test`-এর shape print করো।

In [32]:
# YOUR CODE HERE
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)
X_train.shape, X_test.shape

((712, 13), (179, 13))

---
### Problem 5 — Outlier Handling

**Age column — Z-score method (rows remove করো):**
- `Z_score = (Age - mean) / std`
- যেসব row-এ `|Z_score| > 3`, সেগুলো `X_train` এবং `y_train` থেকে বাদ দাও

**Fare column — IQR clipping (rows রাখো, values clip করো):**
- `Q1` = 25th percentile, `Q3` = 75th percentile
- `IQR = Q3 - Q1`
- `minimum = max(0, Q1 - 1.5 * IQR)`
- `maximum = Q3 + 1.5 * IQR`
- `clip()` দিয়ে Fare column-কে `[minimum, maximum]` range-এ রাখো

In [33]:
# YOUR CODE HERE

Z_score = (X_train['Age'] - X_train['Age'].mean()) / X_train['Age'].std()
X_train = X_train[abs(Z_score) <= 3]
y_train = y_train[abs(Z_score) <= 3]

q1 = X_train['Fare'].quantile(0.25)
q3 = X_train['Fare'].quantile(0.75)
iqr = q3 - q1
minimum = max(0, q1 - 1.5 * iqr)
maximum = q3 + 1.5 * iqr
X_train['Fare'] = X_train['Fare'].clip(lower = minimum, upper = maximum)

---
### Problem 6 — Numerical Pipelines

দুটি numerical pipeline তৈরি করো:

**p1** — `Age` column-এর জন্য:
- Step 1: `SimpleImputer(strategy='mean')`
- Step 2: `StandardScaler()`

**p2** — `Fare` এবং `Family_Size` column-এর জন্য:
- Step 1: `SimpleImputer(strategy='median')`
- Step 2: `MinMaxScaler()`

In [34]:
# YOUR CODE HERE

age_transformer = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'mean')),
    ('scaler', StandardScaler())
])

fare_transformer = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'median')),
    ('scaler', MinMaxScaler())
])

---
### Problem 7 — Categorical Pipelines and ColumnTransformer

দুটি categorical pipeline তৈরি করো:

**p3** — `Embarked`, `Sex`, `Deck` column-এর জন্য:
- Step 1: `SimpleImputer(strategy='most_frequent')`
- Step 2: `OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')`

**p4** — `Pclass` column-এর জন্য:
- `categories = [['third', 'second', 'first']]`
- Step 1: `SimpleImputer(strategy='most_frequent')`
- Step 2: `OrdinalEncoder(categories=categories)`
- Step 3: `MinMaxScaler()`

তারপর `ColumnTransformer` দিয়ে `preprocessor` তৈরি করো:
- `pipeline_1` → `p1` → `['Age']`
- `pipeline_2` → `p2` → `['Fare', 'Family_Size']`
- `pipeline_3` → `p3` → `['Embarked', 'Sex', 'Deck']`
- `pipeline_4` → `p4` → `['Pclass']`
- `remainder='drop'`

In [41]:
# YOUR CODE HERE
categorical_transformer = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('encoder', OneHotEncoder(sparse_output = False, drop = 'first', handle_unknown = 'ignore'))
])

# categories = [['third','second','first']]
categorical_transformer2 = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    # ('encoder', OrdinalEncoder(categories = categories)),
    ('scaler', MinMaxScaler())
])


preprocessor = ColumnTransformer(transformers = [
    ('pipeline_1', age_transformer, ['Age']),
    ('pipeline_2', fare_transformer, ['Fare', 'Family_Size']),
    ('pipeline_3', categorical_transformer, ['Embarked', 'Sex', 'Deck']),
    ('pipeline_4', categorical_transformer2, ['Pclass'])
], remainder = 'drop')

preprocessor

ColumnTransformer(transformers=[('pipeline_1',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('scaler', StandardScaler())]),
                                 ['Age']),
                                ('pipeline_2',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', MinMaxScaler())]),
                                 ['Fare', 'Family_Size']),
                                ('pipeline_3',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['Embarked', 'Sex', 'Deck']),
                                ('pipeline_4',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('scaler', MinMaxScaler())]),
                                 ['Pclass'])])

---
### Problem 8 — Target Column Encoding

`y_train` এবং `y_test`-এ এখন `"yes"` এবং `"no"` string আছে। Model train করার আগে এগুলো numeric করতে হবে।

- `LabelEncoder` দিয়ে `y_train` এবং `y_test` encode করো
- encode করার আগে এবং পরে unique values print করো

In [42]:
# YOUR CODE HERE
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

print(np.unique(y_train))
print(np.unique(y_test))

[0 1]
[0 1]


---
### Problem 9 — Model Training and Prediction

`lr_model` নামে একটি final `Pipeline` তৈরি করো:
- Step 1: `preprocessor` (Problem 7-এ তৈরি করা)
- Step 2: `LogisticRegression(class_weight='balanced', max_iter=1000)`

তারপর:
- `lr_model.fit()` দিয়ে `X_train` এবং `y_train` দিয়ে model train করো
- `predict()` দিয়ে `X_test`-এর prediction করো, `y_pred` নামে save করো
- প্রথম 10টি prediction print করো

In [43]:
# YOUR CODE HERE
lr_model = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced',max_iter=1000))
])

lr_model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('pipeline_1',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age']),
                                                 ('pipeline_2',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   MinMaxScaler())]),
                                                  ['Fare', 'Family_Size']),
                                                 ('pipeline_3',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Embarked', 'Sex', 'Deck']),
                                                 ('pipeline_4',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('scaler',
                                                                   MinMaxScaler())]),
                                                  ['Pclass'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [44]:
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_test)
print(y_pred[:10])

[0 0 0 0 1 1 1 0 1 0]


---
### Problem 10 — Evaluation

`sklearn.metrics` থেকে `accuracy_score`, `precision_score`, `recall_score` import করো।

`y_test` এবং `y_pred` দিয়ে তিনটি score calculate করো এবং নিচের format-এ print করো:

```
Accuracy  : 0.xx
Precision : 0.xx
Recall    : 0.xx
```

In [45]:
# YOUR CODE HERE

from sklearn.metrics import accuracy_score, precision_score, recall_score

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print(f'Accuracy : {accuracy:.2f}')
print(f'Precision : {precision:.2f}')
print(f'Recall : {recall:.2f}')

Accuracy : 0.77
Precision : 0.68
Recall : 0.75
